### Построение нейронной сети на табличных данных

In [3]:
# установим последнюю версию wandb
!pip install -q --upgrade wandb
!pip install tqdm

In [5]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 5.3 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 4.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 5.2 MB/s  0:00:00
  Attempting uninstall: setuptools━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [sympy]
    Found existing installation: setuptools 82.0.1━━━━━━━━━━━━ 1/9 [sympy]
    Uninstalling setuptools-82.0.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [sympy]
      Successfully uninstalled setuptools-82.0.1━━━━━━━━━━━━━━━━━━ 2/9 [setuptools]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [torch]32m8/9 [torch]]x]s]


In [7]:
!pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 5.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]


In [9]:
!pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 5.9 MB/s  0:00:01m0:00:0100:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 8.0 MB/s  0:00:00 eta 0:00:01
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]


In [13]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 7.6 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [14]:
import os
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tqdm import tqdm

Зафиксируем сиды (как в семинаре)

In [15]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything(42)

Зададим класс конфигурации

In [ ]:
class CFG:
    seed = 42
    batch_size = 1024
    num_epochs = 10
    lr = 0.001
    test_size = 0.2

Загрузим нашим данные из файла

In [17]:
df = pd.read_csv("/Users/main/Desktop/Deep learning/Deep-Learning/data/extracted/train_extracted.csv")

df.head()

,Unnamed: 0,sample_id,catalog_content,image_link,price,text_length,word_count,sentence_count,bullet_count,number_count,...,is_food,is_drink,is_health,is_beauty,is_pet,is_baby,is_household,is_premium,is_natural,total_amount
0,0,33127,"item name: la victoria green taco sauce mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,91,19,1,0,3,...,0,0,0,0,0,0,0,0,0,432.00
1,1,198967,"item name: salerno cookies, the original butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,511,81,1,5,11,...,0,0,0,0,0,0,0,0,0,128.00
2,2,261251,"item name: bear creek hearty soup bowl, creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,328,61,2,5,9,...,0,0,0,1,0,0,0,0,0,68.40
3,3,55858,item name: judee’s blue cheese powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,1318,213,9,5,8,...,0,0,0,1,1,0,0,0,0,11.25
4,4,292686,"item name: kedem sherry cooking wine, 12.7 oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,155,29,5,0,5,...,0,0,0,0,0,0,0,0,0,12.00


In [19]:
df.shape

(75000, 60)

Зададим целевую переменную

In [20]:
target = "price"

Уберем все нечисловые признаки, оставим только числовые. Пропуски и пустые места заполним нулями. Для начала попробуем построить модель на них.

In [22]:
X = df.drop(columns=["price", "sample_id", "catalog_content", "image_link", "item_name", "brand", "made_in_country", 
                     "unit"])
X = X.select_dtypes(include=["int64", "float64", "int32", "float32"])
X = X.fillna(0)

y = df[target].values

Далее прологорифмируем цену (так как на этапе EDA было выявленно, что у цены большой разброс). Разделим выборки на тренировочную и валидационную. 